# Week 1: NumPy Basics for Computer Vision

**Computer Vision Fundamentals** — Temasek Poly IIT AAI

*(Student copy — fill in the code where you see **Task** or placeholder comments.)*

Images in OpenCV are **NumPy arrays**. This notebook covers NumPy operations you will use again and again in CV: reshaping, stacking, conditional logic (`np.where`), clipping, and masking.

---
## 1.1 Setup and why shape matters

In computer vision, an image is a 2D (grayscale) or 3D (colour) NumPy array with **shape** `(height, width)` or `(height, width, channels)`. Knowing and changing shape is essential: you might **flatten** a patch to feed a classifier, **reshape** a list of pixels back into an image, or add a **batch dimension** for deep learning (e.g. from `(H, W, C)` to `(1, H, W, C)`).

- **`arr.shape`** gives the dimensions as a tuple. Use it to get height, width, or to check that two images match before concatenating.
- **`arr.reshape(new_shape)`** returns a new view (when possible) with the given dimensions. The total number of elements must stay the same. Using **`-1`** for one dimension lets NumPy infer it from the rest (e.g. `reshape(-1)` flattens to 1D; `reshape(3, -1)` keeps 3 rows and infers the number of columns).

**Task:** Create a small 2D array (e.g. 2×3), print its shape, then use `.reshape(-1)` to flatten it and `.reshape(3, 2)` to change to 3×2.

In [ ]:
import numpy as np

# Your code: create a 2D array, print shape, flatten with reshape(-1), reshape to (3,2)



---
## 1.2 Indexing and slicing — ROIs and regions

**Why this matters in CV:** You often need to work on a **region of interest (ROI)**—a rectangular patch of the image (e.g. a detected face, a license plate, or a crop for further processing). NumPy indexing and slicing is how you select that region.

**Convention:** For a 2D array (grayscale image), the first index is the **row** (vertical, **y**), the second is the **column** (horizontal, **x**). So `img[y, x]` is the pixel at row `y`, column `x`. For colour (3D): `img[y, x, channel]`.

- **Single element:** `arr[row, col]`. **Full row/column:** `arr[row, :]`, `arr[:, col]`.
- **ROI (rectangle):** `arr[r1:r2, c1:c2]` — rows from r1 to r2-1, columns from c1 to c2-1. Example: 100×100 patch at (50, 80) is `img[50:150, 80:180]`.
- **Negative indices:** `-1` = last; e.g. `img[-1, :]` = last row, `img[:, -100:]` = last 100 columns.
- **Step:** `img[::2, ::2]` = every 2nd pixel (downsampling). Slicing returns a **view**; use `.copy()` when you need an independent ROI.

**Task:** Create a 2D array (e.g. 4×5). Print a single pixel `img[1, 2]`, one full row, one full column, and an ROI slice (e.g. rows 1–3, cols 2–5). Try the last row and last two columns using negative indices.

In [ ]:
# Your code: 2D array (e.g. 4×5); print single pixel img[1,2], one row, one column
# ROI = img[1:3, 2:5]; print ROI, last row img[-1,:], last two cols img[:,-2:]



---
## 1.3 np.concatenate — join arrays along an axis

**Application:** Stitch two or more images together, or build a batch of images. For example, place two images of the same height **side by side** (concatenate along columns, `axis=1`) or **one above the other** (along rows, `axis=0`). In deep learning, you often stack many images along a new batch axis.

**Syntax:** `np.concatenate((a, b, ...), axis=...)`. The arrays must have the **same shape on every axis except the one you are concatenating along**. For 2D: `axis=0` = vertical (more rows), `axis=1` = horizontal (more columns). For a 3D image `(H, W, C)`, `axis=0` or `axis=1` still stack in the spatial dimensions; `axis=2` would stack along channels (e.g. adding an extra channel). If shapes don't match, NumPy raises an error.

**Task:** Create two 2×2 arrays. Use `np.concatenate(..., axis=0)` for vertical stack and `axis=1` for horizontal stack; print both results.

In [ ]:
# Your code: two 2D arrays, concatenate axis=0 and axis=1, print



---
## 1.4 np.vstack and np.hstack — stack images into grids

**Application:** These are convenient shortcuts for stacking arrays **vertically** (`np.vstack`) or **horizontally** (`np.hstack`). In CV you use them to build a single image from several smaller ones—e.g. show original, filtered, and masked versions in one row, or stack multiple detected patches into a grid for visualization or debugging.

**Syntax:** `np.vstack((a, b, ...))` is equivalent to `np.concatenate((a, b, ...), axis=0)`; `np.hstack((a, b, ...))` is `axis=1`. You pass a **sequence** (tuple or list) of arrays. All arrays must have the same shape along the non-stacking dimension (for `vstack`, same number of columns; for `hstack`, same number of rows).

**Task:** Create two arrays of the same shape (e.g. 2×3). Use `np.vstack` and `np.hstack` to stack them; print both results.

In [ ]:
# Your code: two same-shape arrays, np.vstack and np.hstack, print



---
## 1.5 np.where — conditional values (thresholds, masks)

**Application:** In CV you often need to **threshold** pixels (e.g. bright = 255, dark = 0 for a binary mask) or **replace values** only where a condition holds. `np.where(condition, value_if_true, value_if_false)` returns an array of the same shape as `condition`: wherever the condition is True it takes `value_if_true`, otherwise `value_if_false`. The condition is typically a comparison on an array (e.g. `img > 128`), which gives a boolean array. You can use this for simple binarization, for building masks to use elsewhere, or for combining two images based on a mask.

**Task:** Create a 1D array of numbers. Use `np.where(arr > 128, 255, 0)` to binarize. Then create a small 2D array and use `np.where` to get a mask (1 where value > 128, 0 otherwise); print the mask.

In [ ]:
# Your code: 1D array, np.where(>128, 255, 0); 2D array, np.where mask, print



---
## 1.6 np.clip — keep values in range

**Application:** When you do arithmetic on pixel values (e.g. add 50 for brightness, or multiply by 1.5), results can go **below 0** or **above 255**. For 8-bit images we must keep values in the range **[0, 255]**. If you don't clip, casting to `uint8` causes **overflow**: values above 255 wrap around (e.g. 260 becomes 4), which can create visible artifacts.

**Syntax:** `np.clip(arr, min_val, max_val)` returns an array (same shape as `arr`) with every element forced into `[min_val, max_val]`. Values below the minimum become the minimum; values above the maximum become the maximum. The typical pipeline after brightness/contrast is: work in float → `np.clip(arr, 0, 255)` → `.astype(np.uint8)`.

**Task:** Create an array with values outside 0–255 (e.g. [-10, 50, 200, 300]). Use `np.clip(arr, 0, 255)` and print the result.

In [ ]:
# Your code: array with values outside [0,255], np.clip(0, 255), print



---
## 1.7 Boolean indexing and masks

**Application:** A **boolean mask** is an array of the same shape as your image with `True`/`False` values. You can use it to **select** only the pixels where the mask is True, or to **assign** new values only to those pixels. For example: get all pixel values in a segmented region, set background to black, or apply a filter only inside a mask.

**Syntax:** `arr[mask]` returns a **1D array** of all elements where `mask` is True (in row-major order). Assigning with `arr[mask] = value` (or `arr[mask] = other_values`) changes only those positions. The mask must be boolean and the same shape as `arr`. This is different from slicing: the result of `arr[mask]` is a flattened selection, not a 2D subregion.

**Task:** Create a 2D array and a boolean mask (e.g. `arr % 2 == 0`). Print `arr[mask]`. Then copy the array, set `arr_mod[mask] = 0`, and print the result.

In [ ]:
# Your code: 2D array, boolean mask, arr[mask], then copy and set mask positions to 0



---
## 1.8 np.zeros_like, np.ones_like — same shape as another array

**Application:** You often need a new array that **matches the shape (and dtype)** of an existing one—e.g. a mask of zeros to fill in, an accumulator, or an output buffer. `np.zeros_like(arr)` and `np.ones_like(arr)` create arrays of zeros or ones with the same shape and dtype as `arr`. This avoids writing `np.zeros(img.shape, dtype=img.dtype)` and keeps your code correct if the image size or type changes.

**Task:** Create a reference 2D array. Use `np.zeros_like(ref)` and `np.ones_like(ref)`; print both and confirm they have the same shape and dtype as `ref`.

In [ ]:
# Your code: ref array, zeros_like(ref), ones_like(ref), print



---
## 1.9 Summary

These operations form the core of array manipulation for CV:

- **shape / reshape**: Inspect and change dimensions; `-1` infers one dimension.
- **Indexing and slicing**: `img[y, x]`, `img[y1:y2, x1:x2]` for ROI; rows = first index, cols = second. Use `.copy()` when you need an independent ROI.
- **np.concatenate**: Join arrays along a given axis (0=vertical, 1=horizontal).
- **np.vstack / np.hstack**: Stack arrays vertically or horizontally (convenience for image grids).
- **np.where**: Conditional values (thresholds, binarization, masks).
- **np.clip**: Keep values in [min, max] (e.g. 0–255 for pixels).
- **Boolean indexing**: Select or set elements where a condition is true.
- **np.zeros_like / np.ones_like**: Create same-shape arrays for masks or buffers.